Notebook for the analysis of the trajectories. <br>
It outputs dataframes (csv files) with the data included in the paper.

In [ ]:
import numpy as np
import pandas as pd
import MDAnalysis as mda
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
from nanover.mdanalysis import universes_from_recording, universe_from_recording
from nanover.recording.reading import NanoverRecordingReader, iter_full_view

def first_chars(x):
    return(x)

In [ ]:
# Load trajectories of glucose, galactose or fructose

file_list_1 = glob.glob(os.path.join('.', "../recordings/glucose/geom*.nanover.zip"))

In [ ]:

# Create empty lists to append simulation data to
frames = []                        # Frame indices
simulation_times = []              # Time elapsed in simulation time
timestamps = []                    # Timestamps for synchronising trajectory and state data
user_forces = []                   # iMD forces acting on the particles 
temperature = []                    #Temperaature
ne_work = []
cm_particles = [159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182]
carbons_rings =  [46, 47, 50, 55, 58, 79, 3, 4, 7, 24, 27, 2] # index of the carbons of the top and bottom aromatic rings
#carbon rings of the receptor "arms", opposite to the direction of unbinding
P1_opp = [20, 21, 12, 13, 14, 15]
P2_opp = [63, 64, 65, 70, 71, 72]
P3_opp = [40, 41, 32, 33, 34, 36]
P_opp_list = [P1_opp, P2_opp, P3_opp]
cm_position_trajs = []
cavity_position_trajs = []
p_position_traj = []
user_magn_forces = []
distances = []


for x in file_list_1:
    print(x)
    u = universe_from_recording(x)
    u.guess_TopologyAttrs(to_guess=['masses'])
    p = int(x.partition("P")[2][0])
    P_opp = P_opp_list[p - 1]
    frames_i = []                        # Frame indices
    simulation_times_i = []              # Time elapsed in simulation time
    timestamps_i = []                    # Timestamps for synchronising trajectory and state data
    user_forces_i = []                   # iMD forces acting on the particles 
    temperature_i = []                    #Temperature


    cm_mass = mda.AtomGroup(cm_particles, u)
    cavity_mass = mda.AtomGroup(carbons_rings, u)
    p_mass = mda.AtomGroup(P_opp, u)
    distance = []

    # Iterate through simulation frames and retrieve simulation data, skipping the first frame
    for timestep in u.trajectory:
        if timestep.frame != 0:
            frames_i.append(timestep.frame)
            simulation_times_i.append(timestep.time)
            timestamps_i.append(int(timestep.data["elapsed"]))
            user_forces_i.append(timestep.data["user_forces"])
            temperature_i.append(timestep.data["system.temperature"]) 
            cm_position = (cm_mass.center_of_mass()) # GET CM POSITION SUGAR DURING TRAJECTORY
            cavity_position = (cavity_mass.center_of_mass()) #GET CAVITY POSITION DURING TRAJECTORY
            p_position = (p_mass.center_of_mass()) #GET carbon group POSITION DURING TRAJECTORY
            v = np.subtract(p_position, cavity_position)
            v1 = np.subtract(cm_position, cavity_position)
            proj = - np.dot(v1, v)/(np.linalg.norm(v))
            distance.append(proj)
    

        
    frames.append(frames_i)
    simulation_times.append(simulation_times_i)
    timestamps.append(timestamps_i)
    user_forces.append(user_forces_i)
    temperature.append(temperature_i)
    distances.append(distance)


    last_path = None
    position_i = []
    timestamps_state_i = []

    for time, frame, state in iter_full_view(x):
            if time in timestamps_i:
                        if frame is not None and state is not None:
                            timestamps_state_i.append(time)
                            interaction_updated = [key for key in state if key.startswith("interaction")]
        
                            
                            if interaction_updated:
                                   position_i.append(np.array(state[interaction_updated[0]]['position'])) # in nm
                            else:
                                   position_i.append([np.nan,np.nan,np.nan])

    position_i = np.array(position_i)

# work with integration thorugh methods of rectangles
# W = sum(N, i=1) F_i v_i,i-1 dt_i,i-1


    forces_i = []
    ne_work = []
    
    #ds user vector
    ds_vector = [(position_i[v] - position_i[v-1]) for v in range(1, len(position_i))] #nm
    # user average velocity fo each interval
    dt = [ (simulation_times_i[x] - simulation_times_i[x-1]) for x in range(1, len(simulation_times_i))] #ps
    velocities = [ds_vector[i]/dt[i] for i in range(len(ds_vector))]  #nm/ps
    velocities.insert(0,np.zeros((3))) # first f_1 velocities is zero
    dt.insert(0,0) # to adjust the size of the arrays
    cum_work = 0
    ne_work.append(cum_work) #initial 0 value
    for p in range(1, len(position_i)):
            cm_force = (np.sum([user_forces_i[p-1][k] for k in cm_particles], axis=0))
            forces_i.append(cm_force)
            
            w = np.dot(cm_force, velocities[p])*dt[p]
            if np.isnan(w) == True  :
                  w = 0.0
            cum_work += w
            ne_work.append(cum_work)

    user_magn_forces.append(forces_i)

    distances = []

    for i in range(0, len(cm_position_trajs)):
        distance = []
        print(i)
        for y in range(0, len(cm_position_trajs[i])):
            v = np.subtract(p_position_traj[i][y], cavity_position_trajs[i][y])
            v1 = np.subtract(cm_position_trajs[i][y], cavity_position_trajs[i][y])
            proj = - np.dot(v1, v)/(np.linalg.norm(v))
            distance.append(proj)
        distances.append(distance)


In [ ]:
for i in range(len(file_list_1)):
    df = pd.DataFrame({
        "frames": frames[i],
        "simulation_times": simulation_times[i],
        "timestamps": timestamps[i],
        "user_forces": user_magn_forces[i],
        "ne_work": ne_work[i],
        "temperature": temperature[i],
       "reaction_coordinate2": distances[i],
    })
    filename = '{}_analysis.csv'.format(file_list_1[i])
    df.to_csv(filename, index=False)

## Potential energy of the complex

In [ ]:
def select_cvs( folder_path ):
    # List all CSV files in the folder
    file_list = glob.glob(os.path.join(folder_path, "*analysis.csv"))
    file_list = sorted(file_list, key=first_chars)
    # Load trajectory back instead of traj file 
    dfs = []
    print(file_list)
    for i in (file_list):
        df = pd.read_csv('{}'.format(i))
        df['name'] = None
        df.iloc[0, df.columns.get_loc('name')] = i  # Only the first row gets the name
        dfs.append(df)
    return dfs

Use our class to extract the potential energy of the sugar-Gluhut complex.

In [ ]:
from extractEnergy import *
for i, x in enumerate(file_list_1):
    print(x)
    u = universe_from_recording(x)
    selection = u.select_atoms('all')
    selection.write('../recordings/galactose/xtc_format/{}.xtc'.format(x), frames='all')

file_xtc = glob.glob(os.path.join('.', "../recordings/galactose/geom*.nanover.zip"))
file_xtc= sorted(file_xtc, key = first_chars) 
file_data_frame = select_cvs("./data_frames/glucose/")
for i, path in enumerate(file_xtc):
    print(path)
    traj_full = md.load(path, top="./0GB_files/glucose_trimmed2.parm7")
    # Load input files
    inpcrd = AmberInpcrdFile("./0GB_files/glucose_trimmed2.rst7")
    prmtop = AmberPrmtopFile("./0GB_files/glucose_trimmed2.parm7", periodicBoxVectors=inpcrd.boxVectors)
    systemm = prmtop.createSystem(
        nonbondedMethod=PME,
        nonbondedCutoff=0.8 * unit.nanometer,
        constraints=HBonds
    )

    system_paramss = {
        "nonbondedMethod": PME,
        "nonbondedCutoff": 0.8 * unit.nanometer,
        "constraints": HBonds,
    }
    integrator = (
        300 * unit.kelvin,
        1 / unit.picosecond,
        0.002 * unit.picoseconds
    )


    enesel_subset = EnergySelect(
        topology_openmm=prmtop.topology,
        system=systemm,
        system_params= system_paramss,
        integrator_params= integrator,
        selection_md="resid 0 or resid 1"
    )
    energy_values_subset = [enesel_subset.calc_energy(frame.xyz).value_in_unit(unit.kilojoule_per_mole) for frame in traj_full]

    #sub_potential.append(energy_values_subset)
    file_data_frame["PE_MOL_0GB"] = energy_values_subset
    """
    This following part was used but not included in the paper. 
    Its purpose is to analyse the potential energy derived from the electrostatic and VanDerWalls term.

    enesel_subset = EnergySelect(
        topology_openmm=prmtop.topology,
        system=systemm,
        system_params= system_paramss,
        integrator_params= integrator,
        selection_md="resid 0"
    )
    energy_values_subset = [enesel_subset.calc_energy(frame.xyz).value_in_unit(unit.kilojoule_per_mole) for frame in traj_full]

    #sub_potential.append(energy_values_subset)
    file_data_frame["PE_MOL"] = energy_values_subset

    enesel_subset = EnergySelect(
        topology_openmm=prmtop.topology,
        system=systemm,
        system_params= system_paramss,
        integrator_params= integrator,
        selection_md="resid 1"
    )
    energy_values_subset = [enesel_subset.calc_energy(frame.xyz).value_in_unit(unit.kilojoule_per_mole) for frame in traj_full]

    #sub_potential.append(energy_values_subset)
    file_data_frame["PE_0GB"] = energy_values_subset


    pe_interaction = [file_data_frame["PE_MOL_0GB"][x] - file_data_frame["PE_0GB"] - file_data_frame["PE_MOL"] for x in range(0, len(file_data_frame["PE_0GB"]))]
    df["PE_int"] = pe_interaction
    """
    
    df.to_csv(df["name"][0])